# Python Lesson 47: Backpropagation

**Concept page:** `wiki/concepts/backpropagation.md` · **Area:** ml-connection

**You will be able to:**
- implement a 2-2-1 network forward pass
- derive gradients with the chain rule
- verify them with a numerical gradient check

*How to use:* run the cells top to bottom, read the output, then try the **Exercises** in the empty cells before opening the **Solutions** section at the bottom.

In [1]:
import numpy as np
import sympy as sp
import matplotlib
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)
%matplotlib inline

## 1. Forward pass

In [2]:
sig=lambda z:1/(1+np.exp(-z))
rng=np.random.default_rng(0)
W1=rng.normal(size=(2,2)); b1=np.zeros(2); W2=rng.normal(size=(2,1)); b2=np.zeros(1)
x=np.array([0.5,-1.0]); y=1.0
def forward(W1,b1,W2,b2,x):
    z1=x@W1+b1; a1=sig(z1); z2=a1@W2+b2; yh=sig(z2); return z1,a1,z2,yh
loss=lambda yh:-(y*np.log(yh)+(1-y)*np.log(1-yh))
print(forward(W1,b1,W2,b2,x)[-1], loss(forward(W1,b1,W2,b2,x)[-1]))

[0.4932] [0.7068]


## 2. Backward pass (chain rule)

In [3]:
def backward(W1,b1,W2,b2,x,y):
    z1,a1,z2,yh=forward(W1,b1,W2,b2,x)
    dz2=yh-y                       # dL/dz2 for sigmoid + log-loss
    dW2=np.outer(a1,dz2); db2=dz2
    da1=W2@dz2; dz1=da1*a1*(1-a1)  # sigmoid'
    dW1=np.outer(x,dz1); db1=dz1
    return dW1,db1,dW2,db2
g=backward(W1,b1,W2,b2,x,y); print(g[0])

[[ 0.0313 -0.0227]
 [-0.0625  0.0455]]


## 3. Gradient check

In [4]:
def L(W1_,b1_,W2_,b2_): return loss(forward(W1_,b1_,W2_,b2_,x)[-1]).item()
eps=1e-6; num=np.zeros_like(W1)
for i in range(2):
    for j in range(2):
        P=W1.copy(); P[i,j]+=eps; M_=W1.copy(); M_[i,j]-=eps
        num[i,j]=(L(P,b1,W2,b2)-L(M_,b1,W2,b2))/(2*eps)
print(np.allclose(num,g[0],atol=1e-6))

True


## 4. Train on XOR with a hidden layer

In [5]:
X=np.array([[0,0],[0,1],[1,0],[1,1]],float); Y=np.array([0,1,1,0.])
rng=np.random.default_rng(1); V1=rng.normal(size=(2,4)); c1=np.zeros(4); V2=rng.normal(size=(4,1)); c2=np.zeros(1)
for _ in range(20000):
    A1=sig(X@V1+c1); P=sig(A1@V2+c2)[:,0]
    dz2=(P-Y)[:,None]; dV2=A1.T@dz2/4; dc2=dz2.mean(0)
    dz1=(dz2@V2.T)*A1*(1-A1); dV1=X.T@dz1/4; dc1=dz1.mean(0)
    V1-=1.0*dV1; c1-=1.0*dc1; V2-=1.0*dV2; c2-=1.0*dc2
print(np.round(P,3))   # ~ [0,1,1,0]

[0.    1.    1.    0.001]


## Cambodian textbook corner (កម្មវិធីសិក្សាកម្ពុជា)
Grade 12 basic chain rule (dy/dx = dy/du · du/dx) is exactly the rule applied layer by layer above.

In [6]:
print('chain rule demo:', 2*0.5*np.cos(0.5**2-3))

chain rule demo: -0.9243023786324636


## Exercises

**Exercise 1.** Run the numerical gradient check for the output weights W2 and compare with the analytic dW2.

In [7]:
# your code here

**Exercise 2.** Symmetry problem: initialise all weights equal (W1=0.3, W2=0.5) and show the two hidden units receive identical gradients, so they can never become different.

In [8]:
# your code here

## Solutions
*(Try the exercises first!)*

In [9]:
# Solution 1
num=np.zeros_like(W2)
for i in range(W2.shape[0]):
    for j in range(W2.shape[1]):
        P_=W2.copy(); P_[i,j]+=eps; M2=W2.copy(); M2[i,j]-=eps
        num[i,j]=(L(W1,b1,P_,b2)-L(W1,b1,M2,b2))/(2*eps)
assert np.allclose(num,g[2],atol=1e-6); print('W2 gradient check passed')

W2 gradient check passed


In [10]:
# Solution 2
W1s=np.full((2,2),0.3); W2s=np.full((2,1),0.5); gs=backward(W1s,np.zeros(2),W2s,np.zeros(1),x,y)
assert np.allclose(gs[0][:,0],gs[0][:,1]); print(gs[0])

[[-0.024 -0.024]
 [ 0.048  0.048]]
